# 🚀 ULM-1.7B QLoRA SFT (Google Colab Training)

이 노트북은 **Qwen3-1.7B** 모델을 구축된 울산 방언 데이터셋(Ulsan Core)으로 파인튜닝하는 Google Colab 전용 실행 노트북입니다.

### 📌 사전 준비
1. 상단 메뉴의 **[런타임] -> [런타임 유형 변경]**에서 하드웨어 가속기를 **T4 GPU** (또는 A100/L4)로 설정하세요.
2. 로컬에서 생성된 `ulsan_dataset.zip` 파일을 Colab 왼쪽 파일 패널에 업로드하세요.

In [ ]:
# 0. ULM repo 확보 (ipynb 단독 실행 시 필수)
import os
if os.path.exists("scripts/train_sft.py"):
    print("✓ repo 루트에서 실행 중")
elif os.path.exists("ULM-1.7B/scripts/train_sft.py"):
    print("✓ ULM-1.7B 폴더 발견, 이동")
    %cd ULM-1.7B
else:
    print("⬇️ ULM-1.7B 클론 중...")
    !git clone https://github.com/UlsanLM-LAB/ULM-1.7B.git
    %cd ULM-1.7B


In [ ]:
# 1. GPU 하드웨어 확인
!nvidia-smi

In [ ]:
# 2. 필수 라이브러리 및 ULM 패키지 설치
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate
!pip install -q -e .

In [ ]:
# 3. 업로드된 ulsan_dataset.zip 압축 해제 (/content vs repo 루트 자동 탐색)
import os, glob, zipfile

dataset_dir = "data/private/ulsan_dataset"
os.makedirs(dataset_dir, exist_ok=True)

cands = glob.glob("ulsan_dataset.zip") + glob.glob("../ulsan_dataset.zip") + glob.glob("/content/*.zip") + glob.glob("/content/ULM-1.7B/*.zip")
zip_path = next((p for p in cands if os.path.basename(p) == "ulsan_dataset.zip" and os.path.exists(p)), None)
print("후보:", cands, "\n선택:", zip_path, "\n현재위치:", os.getcwd())

if zip_path:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        # zip 내부에 data/private/... prefix가 있으면 repo 루트에 풀어야 이중폴더 방지
        if any(n.startswith("data/private/ulsan_dataset/") for n in names):
            z.extractall(".")
        else:
            z.extractall(dataset_dir)
    print(f"✓ 데이터셋 압축 해제 완료: {dataset_dir}")
    !ls -lh {dataset_dir}
elif os.path.exists(f"{dataset_dir}/train.jsonl"):
    print(f"✓ 데이터셋이 이미 존재합니다: {dataset_dir}")
else:
    print("⚠️ 'ulsan_dataset.zip' 파일을 Colab 왼쪽 파일 영역에 업로드해주세요!")
    print("현재위치:", os.getcwd(), "| /content 내용:")
    !ls -lh /content/*.zip 2>&1 | head; ls -lh *.zip 2>&1 | head

In [ ]:
# 4. ULM-1.7B QLoRA SFT 학습 실행 (백그라운드 체크포인트 저장 지원)
!python scripts/train_sft.py --config configs/sft/qwen3_1.7b_qlora.yaml

In [ ]:
# 5. 학습 완료된 어댑터로 울산 방언 추론 테스트
!python scripts/infer.py \
    --base-model Qwen/Qwen3-1.7B \
    --adapter-path outputs/qwen3-1.7b-sft \
    --prompt "오늘 날씨 참 좋다, 저녁에 밥 뭐 먹으러 갈래?" \
    --dialect-strength 2